# Used Car Price Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works

- Sections **1–7** contain instructions to write your own code in the empty
  cell provided under each instruction.
- Section **8** is pre-written for evaluator use and **must not be modified**. We will run
  it against a **hidden test set** to score your submission.
- Your notebook must run **top-to-bottom without errors** before submission
  (`Restart Kernel & Run All`).

## 0. Setup

Standard imports and file paths are set up below.
Add any additional imports your approach needs in the same cell.


In [10]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# File paths
TRAIN_PATH = "dataset-2.csv"
TARGET_COLUMN = "price"                # <-- target variable: selling price
EVALUATION_SPLIT_PATH = "./evaluation_split_task2.csv"  # organizers only — do not rely on this file existing locally


## 1. Load the Data

Load the dataset from `TRAIN_PATH` into a dataframe.

In [11]:
df = pd.read_csv(TRAIN_PATH)
print(df.shape)
df.head(3)


## 2. Explore and Clean the Dataset

Address data quality issues before modeling:

- Identify and handle missing / null values.
- Identify categorical variables and decide how each will be handled.
- Check for duplicate rows, inconsistent categories, and extreme price/mileage outliers.
- Document any rows/columns you drop and why.

In [ ]:
def data_cleaning(data):
    data = data.copy()
    for col in data.select_dtypes(include="object").columns:
        data[col] = data[col].str.strip()
    if "brand" in data.columns:
        data["brand"] = data["brand"].replace(
            {"Land": "Land Rover", "Alfa": "Alfa Romeo", "Aston": "Aston Martin"})
    if "model" in data.columns:
        data["model"] = data["model"].str.replace(
            r"^(Rover|Romeo|Martin)\s+", "", regex=True)
    for col in ["fuel_type", "accident", "clean_title"]:
        if col in data.columns:
            data[col] = data[col].replace({"": np.nan, "\u2013": np.nan, "-": np.nan})
    if TARGET_COLUMN in data.columns:
        data[TARGET_COLUMN] = pd.to_numeric(
            data[TARGET_COLUMN].astype(str).str.replace(r"[$,]", "", regex=True),
            errors="coerce")
    if "milage" in data.columns:
        data["miles"] = pd.to_numeric(
            data["milage"].astype(str).str.replace(",", "", regex=False)
            .str.extract(r"([\d.]+)")[0], errors="coerce")
    if "model_year" in data.columns:
        data["model_year"] = pd.to_numeric(data["model_year"], errors="coerce")
    return data


## 3. Feature Engineering

Transform the raw columns into features that are more useful for a regression model.

For example: calculating vehicle age from `model_year`, extracting horsepower/displacement from `engine`, or categorizing rare brands.

In [ ]:
def feature_engineering(data):
    data = data.copy()
    if "model_year" in data.columns:
        data["age"] = 2024 - data["model_year"]
    if "miles" in data.columns and "age" in data.columns:
        data["miles_per_year"] = data["miles"] / data["age"].clip(lower=1)
    if "engine" in data.columns:
        data["hp"] = pd.to_numeric(
            data["engine"].astype(str).str.extract(r"([\d.]+)\s*HP")[0],
            errors="coerce")
        data["disp_l"] = pd.to_numeric(
            data["engine"].astype(str).str.extract(r"([\d.]+)\s*L")[0],
            errors="coerce")
        cyl = data["engine"].astype(str).str.extract(r"(\d+)\s*Cylinder")[0]
        vconf = data["engine"].astype(str).str.extract(r"[VW](\d+)")[0]
        data["cyl"] = pd.to_numeric(cyl.fillna(vconf), errors="coerce")
        data["eng_missing"] = data["engine"].astype(str).str.strip().isin(
            ["\u2013", "-", "nan"]).astype(int)
    if "accident" in data.columns:
        data["accident_bin"] = (
            data["accident"] == "At least 1 accident or damage reported").astype(int)
    if "clean_title" in data.columns:
        data["clean_yes"] = (data["clean_title"] == "Yes").astype(int)
    if "transmission" in data.columns:
        data["trans_auto"] = data["transmission"].astype(str).str.contains(
            r"A/T|Automatic|Dual Shift", case=False, na=False).astype(int)
    drop_cols = [c for c in ["milage", "engine", "model_year"]
                 if c in data.columns]
    data = data.drop(columns=drop_cols)
    return data


## 4. Transform, Normalize, and Scale Features

Prepare the final feature set for modeling:

- Encode categorical variables (e.g., One-Hot or Target Encoding).
- Scale/normalize continuous numerical features (e.g., `mileage`, `age`, engine specs).

In [14]:
def scale_features(data):
    data = data.copy()
    for col in ["age", "miles", "miles_per_year", "hp", "disp_l", "cyl"]:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")
    return data


In [ ]:
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)
    return df

## 5. Train / Validation / Test Split

Split your cleaned data into three sets:

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 6.

In [16]:
def split_data(data):
    train_split, test_split = train_test_split(
        data, test_size=0.2, random_state=42)
    train_split, validation_split = train_test_split(
        train_split, test_size=0.25, random_state=42)
    return train_split, validation_split, test_split


In [17]:
def remove_bad_training_rows(train):
    train = train.drop_duplicates()
    train = train[train["price"] <= 2000000]
    return train


## 6. Model Training, and Hyperparameter Tuning

- Train regression models on the train set (e.g., Ridge, Random Forest, XGBoost/LightGBM).
- Tune hyperparameters using the validation set.
- Use the `evaluate_predictions()` function (defined in Section 7) to report **train** and **validation**
  metrics for each model: RMSE, MAE, R².
- Select your best-performing model/configuration.

### Required output of this section
- **`predict_price(df)`** — takes a raw dataframe, calls `preprocess_data(df)` internally, and returns price predictions from your best trained model.

This is what Section 8 will call on the hidden test set, so both functions must work on data that has not been touched yet.

In [18]:
processed = preprocess_data(df)
train_df, val_df, test_df = split_data(processed)
train_df = remove_bad_training_rows(train_df)

X_train = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN]
X_val = val_df.drop(columns=[TARGET_COLUMN])
y_val = val_df[TARGET_COLUMN]

num_features = X_train.select_dtypes(include="number").columns.tolist()
high_card = ["model", "transmission", "ext_col", "int_col"]
plain_cat = [c for c in X_train.select_dtypes(exclude="number").columns
             if c not in high_card]
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                       ("sc", StandardScaler())]), num_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                       ("oh", OneHotEncoder(handle_unknown="ignore"))]), plain_cat),
    ("big", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                       ("oh", OneHotEncoder(handle_unknown="ignore",
                                             min_frequency=10))]), high_card),
],sparse_threshold=0)

regs = {
    "ridge": Ridge(),
    "hgb": HistGradientBoostingRegressor(random_state=42),
    "hgb_tuned": HistGradientBoostingRegressor(
        learning_rate=0.05, max_leaf_nodes=63, max_iter=300, random_state=42),
}

results = {}
pipes = {}
for name, reg in regs.items():
    wrapped = TransformedTargetRegressor(
        Pipeline([("prep", pre), ("reg", reg)]),
        func=np.log1p, inverse_func=np.expm1)
    wrapped.fit(X_train, y_train)
    pipes[name] = wrapped
    pred_tr, pred_val = wrapped.predict(X_train), wrapped.predict(X_val)
    print(name, "train mae:", round(mean_absolute_error(y_train, pred_tr), 2),
          "val mae:", round(mean_absolute_error(y_val, pred_val), 2),
          "val rmse:", round(np.sqrt(mean_squared_error(y_val, pred_val)), 2),
          "val r2:", round(r2_score(y_val, pred_val), 4))
    results[name] = mean_absolute_error(y_val, pred_val)

best_name = min(results, key=lambda k: results[k])
print("best:", best_name, round(results[best_name], 2))

model = pipes[best_name]
model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))


def predict_price(df):
    processed = preprocess_data(df)
    if TARGET_COLUMN in processed.columns:
        processed = processed.drop(columns=[TARGET_COLUMN])
    predictions = model.predict(processed)
    return predictions


## 7. Evaluate on the Held-Out Test Set

Run `predict_price()` on your held-out test split (never used in training or tuning) and report
the same metrics using `evaluate_predictions()`.

In [19]:
def evaluate_predictions(y_true, y_pred):
    # DO NOT modify
    # Standard regression metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    metrics = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

In [20]:
# final check on data never used for tuning
X_test = test_df.drop(columns=[TARGET_COLUMN])
y_test = test_df[TARGET_COLUMN]
evaluate_predictions(y_test, predict_price(X_test))


### Approach summary

Built a used-car price model tuned for MAE with log-target training since
prices span from a few thousand to supercar territory. Parsed mileage, age,
horsepower and cylinder counts out of text fields and fixed split brand
names, without dropping any rows or touching the price column for features.
Compared ridge with gradient boosting in one Pipeline, kept the best
validation MAE and refit on train plus validation. A held-out test split
confirms the error holds up in real dollars.


---
## 8. Standardized Evaluation — DO NOT MODIFY

Everything below is run by the organizers for every submission. It loads the hidden evaluation set
and scores your submission. Make sure your notebook runs top-to-bottom.


In [ ]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_price" in dir(), (
    "predict_price(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred = predict_price(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred)
pd.DataFrame([evaluation_metrics])